In [1]:
# ============================================================
# NOTEBOOK 01 — DATA INGESTION & INITIAL EXPLORATION
# E-Commerce Customer Retention & Revenue Optimization Analytics
# ============================================================
# PURPOSE : Load the UCI Online Retail dataset, explore its
#           structure, and save a clean raw backup.
#
# ► FIX FOR "Sales_Data.xlsx" ERROR:
#   Jupyter keeps variables alive between runs. If you previously
#   ran another notebook that set INPUT_PATH to something else,
#   that old value was still in memory. This notebook clears all
#   relevant variables at the top to prevent that.
#
# ► WHERE TO PUT YOUR CSV:
#   Place your downloaded file at exactly this path inside your
#   project folder:
#       Ecommerce_Retention_Analytics/
#           data/
#               raw/
#                   OnlineRetail.csv   ← here
#
#   Your Jupyter notebook must also be launched from the
#   Ecommerce_Retention_Analytics/ folder (not from inside data/).
# ============================================================

import pandas as pd
import os
import glob

# ── CLEAR ANY STALE VARIABLES FROM PREVIOUS NOTEBOOK RUNS ───
# This prevents Jupyter's persistent kernel from using an old
# INPUT_PATH value (e.g. one pointing to Sales_Data.xlsx).
for _var in ["INPUT_PATH", "BACKUP_PATH", "df"]:
    try:
        del globals()[_var]
    except KeyError:
        pass

# ── CONFIGURATION ────────────────────────────────────────────
# If run from notebooks directory, change working directory to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

INPUT_PATH  = os.path.join("data", "raw", "OnlineRetail.csv")
BACKUP_PATH = os.path.join("data", "raw", "OnlineRetail_raw.csv")

print("=" * 60)
print("NOTEBOOK 01: DATA INGESTION & INITIAL EXPLORATION")
print("=" * 60)
print(f"\n[CONFIG] Input  path : {os.path.abspath(INPUT_PATH)}")
print(f"[CONFIG] Backup path : {os.path.abspath(BACKUP_PATH)}")
print(f"[CONFIG] Working dir : {os.getcwd()}")


# ─────────────────────────────────────────────────────────────
# SECTION 1 — LOAD DATASET
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 1: LOADING DATASET")
print("=" * 60)

def _find_csv_fallback(start_dir: str) -> str | None:
    """
    If the expected path doesn't exist, search up to 3 levels deep
    for any file named OnlineRetail.csv so we can give the user
    the exact path they need to use — helpful when they placed the
    file in the wrong subfolder.
    """
    pattern = os.path.join(start_dir, "**", "OnlineRetail.csv")
    matches = glob.glob(pattern, recursive=True)
    return matches[0] if matches else None

try:
    # ── verify the file actually exists before trying to open it ──
    if not os.path.isfile(INPUT_PATH):
        suggestion = _find_csv_fallback(".")
        print(f"\n[ERROR] File not found at:")
        print(f"        {os.path.abspath(INPUT_PATH)}\n")

        if suggestion:
            print(f"[HINT]  Found OnlineRetail.csv at a different location:")
            print(f"        {os.path.abspath(suggestion)}")
            print(f"\n        Either move it to the path above, OR change")
            print(f"        INPUT_PATH in this cell to:")
            print(f"        INPUT_PATH = r\"{suggestion}\"")
        else:
            print("[HINT]  OnlineRetail.csv was not found anywhere in the")
            print("        project folder. Steps to fix:")
            print()
            print("        1. Go to Kaggle and download the dataset:")
            print("           https://www.kaggle.com/datasets/denisexpsitou/")
            print("           uci-machine-learning-online-retail-transactions")
            print()
            print("        2. Extract the zip file.")
            print()
            print("        3. Place OnlineRetail.csv here:")
            print(f"           {os.path.abspath(INPUT_PATH)}")
            print()
            print("        4. Re-run this cell (Kernel → Restart & Run All")
            print("           to also clear the stale variable issue).")
        raise FileNotFoundError(f"OnlineRetail.csv not found at {INPUT_PATH}")

    # ── load — UCI file uses latin-1 (contains £ and similar chars) ──
    print(f"\n[INFO] Reading: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH, encoding="latin-1", low_memory=False)
    print(f"[OK]   Loaded successfully.\n")

except FileNotFoundError:
    raise   # already printed a clear message above

except UnicodeDecodeError:
    print("[ERROR] Encoding error.")
    print("        Try: df = pd.read_csv(INPUT_PATH, encoding='cp1252')")
    raise

except Exception as e:
    print(f"[ERROR] Unexpected error: {e}")
    raise


# ─────────────────────────────────────────────────────────────
# SECTION 2 — ROWS & COLUMNS
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("SECTION 2: SHAPE — ROWS & COLUMNS")
print("=" * 60)

total_rows, total_cols = df.shape
print(f"\n  Total rows    : {total_rows:,}")
print(f"  Total columns : {total_cols}")


# ─────────────────────────────────────────────────────────────
# SECTION 3 — COLUMN NAMES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 3: COLUMN NAMES")
print("=" * 60)

column_descriptions = {
    "InvoiceNo"   : "Invoice number — prefix 'C' means cancellation/return",
    "StockCode"   : "Product code (alphanumeric)",
    "Description" : "Product name/label",
    "Quantity"    : "Units per transaction (negatives = returns)",
    "InvoiceDate" : "Date and time of invoice",
    "UnitPrice"   : "Price per unit in GBP (£)",
    "CustomerID"  : "Unique customer ID — NaN rows are guest checkouts",
    "Country"     : "Country the customer is based in",
}

print(f"\n  {'#':<5} {'Column':<20} {'Dtype':<12} Description")
print("  " + "-" * 72)
for i, col in enumerate(df.columns, 1):
    desc  = column_descriptions.get(col, "—")
    dtype = str(df[col].dtype)
    print(f"  {i:<5} {col:<20} {dtype:<12} {desc}")


# ─────────────────────────────────────────────────────────────
# SECTION 4 — DATA TYPES  (with what to do about each)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 4: DATA TYPES & REQUIRED CONVERSIONS")
print("=" * 60)

conversions = {
    "InvoiceNo"   : "Keep as string — use .str.startswith('C') for cancellations",
    "StockCode"   : "Keep as string — mixed alphanumeric (e.g. '85123A')",
    "Description" : "Keep as string — strip & upper() before grouping",
    "Quantity"    : "Already int — remove rows where Quantity <= 0",
    "InvoiceDate" : "► Convert with pd.to_datetime() in Notebook 02",
    "UnitPrice"   : "Already float — remove rows where UnitPrice <= 0",
    "CustomerID"  : "► Cast to pd.Int64Dtype() after filling NaNs in Notebook 02",
    "Country"     : "Keep as string — strip whitespace before grouping",
}

print(f"\n  {'Column':<20} {'Current dtype':<16} Required action")
print("  " + "-" * 72)
for col in df.columns:
    action = conversions.get(col, "No change needed")
    print(f"  {col:<20} {str(df[col].dtype):<16} {action}")


# ─────────────────────────────────────────────────────────────
# SECTION 5 — MISSING VALUES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 5: MISSING VALUES")
print("=" * 60)

missing_count = df.isnull().sum()
missing_pct   = (missing_count / total_rows * 100).round(2)

action_map = {
    "CustomerID"  : "Assign 'GUEST' — track guest vs. registered conversion",
    "Description" : "Drop rows — product name required for segmentation",
}

print(f"\n  {'Column':<20} {'Missing':<10} {'Missing %':<12} Action")
print("  " + "-" * 72)
for col in df.columns:
    cnt    = missing_count[col]
    pct    = missing_pct[col]
    action = action_map.get(col, "None needed") if cnt > 0 else "—"
    flag   = "  ◄" if cnt > 0 else ""
    print(f"  {col:<20} {cnt:<10,} {str(pct)+'%':<12} {action}{flag}")

cols_with_missing = (missing_count > 0).sum()
print(f"\n  [SUMMARY] {cols_with_missing} column(s) have missing values.")


# ─────────────────────────────────────────────────────────────
# SECTION 6 — FIRST 10 ROWS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 6: FIRST 10 ROWS")
print("=" * 60)

pd.set_option("display.max_colwidth", 30)
pd.set_option("display.width", 120)
print()
print(df.head(10).to_string(index=True))
pd.reset_option("display.max_colwidth")
pd.reset_option("display.width")


# ─────────────────────────────────────────────────────────────
# SECTION 7 — BASIC STATISTICS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 7: BASIC STATISTICS — NUMERIC COLUMNS")
print("=" * 60)

num_cols = ["Quantity", "UnitPrice"]
stats = df[num_cols].agg(["mean", "median", "min", "max", "std"]).T.round(2)
stats.columns = ["Mean", "Median", "Min", "Max", "Std Dev"]

print()
print(stats.to_string())

print("\n  [DATA QUALITY FLAGS]")

neg_qty   = (df["Quantity"] < 0).sum()
zero_qty  = (df["Quantity"] == 0).sum()
zero_price = (df["UnitPrice"] <= 0).sum()
high_price = (df["UnitPrice"] > 1000).sum()

flags = [
    (neg_qty,    f"{neg_qty:,} rows with Quantity < 0  → cancellation/return lines (remove)"),
    (zero_qty,   f"{zero_qty:,} rows with Quantity = 0  → data errors (remove)"),
    (zero_price, f"{zero_price:,} rows with UnitPrice ≤ 0 → free samples or errors (remove)"),
    (high_price, f"{high_price:,} rows with UnitPrice > £1,000 → likely wholesale entries (review)"),
]

found_any = False
for count, msg in flags:
    if count > 0:
        print(f"  ► {msg}")
        found_any = True
if not found_any:
    print("  ✓ No obvious numeric data-quality issues detected.")


# ─────────────────────────────────────────────────────────────
# SECTION 8 — UNIQUE CUSTOMERS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 8: UNIQUE CUSTOMERS")
print("=" * 60)

unique_customers = int(df["CustomerID"].dropna().astype(int).nunique())
guest_rows       = int(df["CustomerID"].isna().sum())
guest_pct        = round(guest_rows / total_rows * 100, 1)
registered_rows  = total_rows - guest_rows

print(f"\n  Unique registered customer IDs : {unique_customers:,}")
print(f"  Rows with a CustomerID         : {registered_rows:,}  ({100 - guest_pct:.1f}%)")
print(f"  Rows WITHOUT a CustomerID      : {guest_rows:,}  ({guest_pct}%)  ← guest checkouts")
print(f"\n  [NOTE] Guest rows will get CustomerID = 'GUEST' in Notebook 02.")
print(f"         This lets us measure guest vs. registered funnel drop-off.")


# ─────────────────────────────────────────────────────────────
# SECTION 9 — UNIQUE PRODUCTS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 9: UNIQUE PRODUCTS")
print("=" * 60)

unique_stockcodes = df["StockCode"].nunique()
unique_descs      = df["Description"].dropna().nunique()
desc_code_diff    = abs(unique_descs - unique_stockcodes)

print(f"\n  Unique StockCodes   : {unique_stockcodes:,}")
print(f"  Unique Descriptions : {unique_descs:,}")

if desc_code_diff > 0:
    print(f"\n  [NOTE] {desc_code_diff:,} gap between codes and descriptions.")
    print(f"         Some products have multiple description variants.")
    print(f"         → Always use StockCode as the canonical product key.")

top5_products = df.groupby("StockCode")["Quantity"].sum().nlargest(5)
print(f"\n  Top 5 products by total quantity sold:")
print(f"  {'StockCode':<15} {'Total Qty':>10}")
print("  " + "-" * 28)
for code, qty in top5_products.items():
    print(f"  {str(code):<15} {qty:>10,}")


# ─────────────────────────────────────────────────────────────
# SECTION 10 — DATE RANGE
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 10: DATE RANGE")
print("=" * 60)

try:
    dates      = pd.to_datetime(df["InvoiceDate"], dayfirst=False, errors="coerce")
    n_bad_date = int(dates.isna().sum())
    if n_bad_date > 0:
        print(f"\n  [WARN] {n_bad_date:,} InvoiceDate values could not be parsed.")

    earliest  = dates.min()
    latest    = dates.max()
    span_days = (latest - earliest).days

    print(f"\n  Earliest invoice : {earliest.strftime('%d %b %Y  %H:%M')}")
    print(f"  Latest invoice   : {latest.strftime('%d %b %Y  %H:%M')}")
    print(f"  Data spans       : {span_days} days  (~{span_days / 30.4:.1f} months)")

    # Monthly transaction volume (quick health check — helps spot
    # months that are truncated or have anomalous drops)
    monthly = dates.dt.to_period("M").value_counts().sort_index()
    print(f"\n  Monthly transaction counts:")
    print(f"  {'Month':<12} {'Transactions':>14}  Volume bar")
    print("  " + "-" * 50)
    max_month = monthly.max()
    for period, count in monthly.items():
        bar_len = int(count / max_month * 30)
        bar     = "█" * bar_len
        flag    = "  ← last month (partial)" if period == monthly.index[-1] else ""
        print(f"  {str(period):<12} {count:>14,}  {bar}{flag}")

    # Top countries
    print(f"\n  Top 5 countries by row count:")
    top_countries = df["Country"].str.strip().value_counts().head(5)
    col_w = max(len(c) for c in top_countries.index) + 2
    for country, cnt in top_countries.items():
        pct = cnt / total_rows * 100
        bar = "█" * int(pct / 2)
        print(f"  {country:<{col_w}} {cnt:>8,}  ({pct:5.1f}%)  {bar}")

except Exception as e:
    print(f"[ERROR] Date analysis failed: {e}")


# ─────────────────────────────────────────────────────────────
# SECTION 11 — SAVE RAW BACKUP
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SECTION 11: SAVING RAW BACKUP")
print("=" * 60)

try:
    os.makedirs(os.path.dirname(BACKUP_PATH), exist_ok=True)

    # Save as UTF-8 — the backup is clean text; the latin-1 encoding
    # was only needed to read the original Kaggle CSV export.
    df.to_csv(BACKUP_PATH, index=False, encoding="utf-8")

    size_kb = os.path.getsize(BACKUP_PATH) / 1024
    size_mb = size_kb / 1024
    size_str = f"{size_mb:.1f} MB" if size_mb >= 1 else f"{size_kb:.0f} KB"

    print(f"\n  [OK] Backup saved → {BACKUP_PATH}")
    print(f"       Size    : {size_str}")
    print(f"       Rows    : {len(df):,}")
    print(f"       Columns : {len(df.columns)}")
    print(f"\n  [NOTE] This is the unmodified source-of-truth file.")
    print(f"         Notebook 02 (Data Cleaning) reads from this backup.")

except Exception as e:
    print(f"[ERROR] Could not save backup: {e}")


# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("INGESTION COMPLETE — SUMMARY")
print("=" * 60)

rows_with_c = df["InvoiceNo"].astype(str).str.startswith("C").sum()

summary_items = [
    ("Total rows (raw)",            f"{total_rows:,}"),
    ("Total columns",               total_cols),
    ("Unique registered customers", f"{unique_customers:,}"),
    ("Guest checkout rows",         f"{guest_rows:,}  ({guest_pct}%)"),
    ("Unique products (StockCode)", f"{unique_stockcodes:,}"),
    ("Date range",                  f"{earliest.strftime('%b %Y')} – {latest.strftime('%b %Y')}"),
    ("Columns with missing values", int((missing_count > 0).sum())),
    ("Cancellation invoices (C…)",  f"{rows_with_c:,}"),
    ("Rows with negative Quantity", f"{neg_qty:,}"),
    ("Rows with UnitPrice ≤ 0",    f"{zero_price:,}"),
    ("Raw backup saved to",         BACKUP_PATH),
]

print()
for label, value in summary_items:
    print(f"  {label:<38} {value}")

print()
print("  ✓ Notebook 01 complete.")
print("  → Next: Run Notebook 02 — Data Cleaning & Preprocessing")
print("=" * 60)

NOTEBOOK 01: DATA INGESTION & INITIAL EXPLORATION

[CONFIG] Input  path : C:\Users\Rajashree\Desktop\Ecommerce_Retention_Analytics\data\raw\OnlineRetail.csv
[CONFIG] Backup path : C:\Users\Rajashree\Desktop\Ecommerce_Retention_Analytics\data\raw\OnlineRetail_raw.csv
[CONFIG] Working dir : C:\Users\Rajashree\Desktop\Ecommerce_Retention_Analytics

SECTION 1: LOADING DATASET

[INFO] Reading: data\raw\OnlineRetail.csv


[OK]   Loaded successfully.

SECTION 2: SHAPE — ROWS & COLUMNS

  Total rows    : 541,909
  Total columns : 8

SECTION 3: COLUMN NAMES

  #     Column               Dtype        Description
  ------------------------------------------------------------------------
  1     InvoiceNo            str          Invoice number — prefix 'C' means cancellation/return
  2     StockCode            str          Product code (alphanumeric)
  3     Description          str          Product name/label
  4     Quantity             int64        Units per transaction (negatives = returns)
  5     InvoiceDate          str          Date and time of invoice
  6     UnitPrice            float64      Price per unit in GBP (£)
  7     CustomerID           float64      Unique customer ID — NaN rows are guest checkouts
  8     Country              str          Country the customer is based in

SECTION 4: DATA TYPES & REQUIRED CONVERSIONS

  Column               Current dtype    Required action
  ---------------


  Unique StockCodes   : 4,070
  Unique Descriptions : 4,223

  [NOTE] 153 gap between codes and descriptions.
         Some products have multiple description variants.
         → Always use StockCode as the canonical product key.

  Top 5 products by total quantity sold:
  StockCode        Total Qty
  ----------------------------
  22197               56,450
  84077               53,847
  85099B              47,363
  85123A              38,830
  84879               36,221

SECTION 10: DATE RANGE



  Earliest invoice : 01 Dec 2010  08:26
  Latest invoice   : 09 Dec 2011  12:50
  Data spans       : 373 days  (~12.3 months)

  Monthly transaction counts:
  Month          Transactions  Volume bar
  --------------------------------------------------
  2010-12              42,481  ███████████████
  2011-01              35,147  ████████████
  2011-02              27,707  █████████
  2011-03              36,748  █████████████
  2011-04              29,916  ██████████
  2011-05              37,030  █████████████
  2011-06              36,874  █████████████
  2011-07              39,518  █████████████
  2011-08              35,284  ████████████
  2011-09              50,226  █████████████████
  2011-10              60,742  █████████████████████
  2011-11              84,711  ██████████████████████████████
  2011-12              25,525  █████████  ← last month (partial)

  Top 5 countries by row count:
  United Kingdom    495,478  ( 91.4%)  █████████████████████████████████████████████
  


  [OK] Backup saved → data\raw\OnlineRetail_raw.csv
       Size    : 46.3 MB
       Rows    : 541,909
       Columns : 8

  [NOTE] This is the unmodified source-of-truth file.
         Notebook 02 (Data Cleaning) reads from this backup.

INGESTION COMPLETE — SUMMARY

  Total rows (raw)                       541,909
  Total columns                          8
  Unique registered customers            4,372
  Guest checkout rows                    135,080  (24.9%)
  Unique products (StockCode)            4,070
  Date range                             Dec 2010 – Dec 2011
  Columns with missing values            2
  Cancellation invoices (C…)             9,288
  Rows with negative Quantity            10,624
  Rows with UnitPrice ≤ 0                2,517
  Raw backup saved to                    data\raw\OnlineRetail_raw.csv

  ✓ Notebook 01 complete.
  → Next: Run Notebook 02 — Data Cleaning & Preprocessing
